# Hands-on — your own documents

You have seen the whole method. Now run it on papers you care about.

Five things to fill in, in order: **where your PDFs are**, a **system prompt**,
a **user prompt**, and a **schema**. Then run the last cell and read the table.

Nothing here is new. The first code block is every function the workshop built,
collected in one place so you do not have to run five sessions again.

## Setup

Same as the workshop: Ollama, the two models, this repository. If you have just
come from `workshop.ipynb` in the same runtime, this is quick — the models are
already on disk.

In [ ]:
import glob, os, subprocess, sys, time

IN_COLAB = "google.colab" in sys.modules
REPO = "2026_UIUC_workshop_llm_pdf_extraction"
BRANCH = "main"   # switch to your working branch to test unmerged changes

if IN_COLAB:
    !DEBIAN_FRONTEND=noninteractive apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pandas pillow
    if os.path.basename(os.getcwd()) != REPO:   # so this cell is safe to re-run
        if not os.path.isdir(REPO):
            !git clone -q --branch {BRANCH} https://github.com/de-Medeiros-insect-lab/{REPO}.git
        os.chdir(REPO)
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import ollama

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

assert server_ready(), "Ollama did not start"
print("Ollama is up")

!ollama pull qwen3.5:9b
!ollama pull deepseek-ocr

## Everything the workshop built

One block, no surprises: the imports, the constants and every function from
`workshop.ipynb`. Run it and read on.

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

REPO = "2026_UIUC_workshop_llm_pdf_extraction"

BRANCH = "main"

import time, base64, json, glob, re

import ollama, pymupdf, pandas as pd

from PIL import Image

from IPython.display import display, Image as ShowImage

import io

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

def gpu_report():
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                             capture_output=True, text=True)
        print("GPU:", smi.stdout.strip() if smi.returncode == 0 else "none")
    except FileNotFoundError:
        print("GPU: none")
    try:
        ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
        rows = [r for r in ps.stdout.strip().splitlines()[1:] if r.strip()]
        print("Ollama is running:", rows or "nothing loaded yet")
    except FileNotFoundError:
        print("Ollama is running: the ollama command was not found -- "
              "did the setup cell above finish?")

CHAT_MODEL = "qwen3.5:9b"

OCR_MODEL  = "deepseek-ocr"

DEFAULT_DPI = 100

def open_pdf(path):
    return pymupdf.open(path)

def get_page_text(doc, page):
    """The text layer already stored inside the PDF. Free and instant."""
    if not 1 <= page <= doc.page_count:
        raise ValueError(f"page {page} out of range (1-{doc.page_count})")
    return doc[page - 1].get_text()

def render_page(doc, page, dpi=DEFAULT_DPI):
    """Draw a page as an image, encoded for sending to a model."""
    if not 1 <= page <= doc.page_count:
        raise ValueError(f"page {page} out of range (1-{doc.page_count})")
    return base64.b64encode(
        doc[page - 1].get_pixmap(dpi=dpi).tobytes("png")).decode()

from IPython.display import Image as ShowImage, display

def ocr_page(doc, page, dpi=DEFAULT_DPI):
    """Transcribe a page from its image, with region coordinates.

    Coordinates come back scaled 0-1000, labelled text / image /
    image_caption. No think= here: this model does not reason, it transcribes.
    """
    reply = ollama.generate(
        model=OCR_MODEL,
        prompt="<image>\n<|grounding|>Convert the document to markdown.",
        images=[render_page(doc, page, dpi=dpi)],
        options={"temperature": 0, "num_predict": 4096},
    )
    return reply.response or ""

def regions(ocr_text):
    """Every labelled region: [(label, x1, y1, x2, y2), ...], scaled 0-1000."""
    found = re.findall(r"(\w+)\s*\[\[\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\s*\]\]",
                       ocr_text)
    return [(lab, *map(int, box)) for lab, *box in found]

def crop_region(doc, page, box, dpi=150, pad=0.01):
    """Cut one 0-1000 box out of a rendered page."""
    im = Image.open(io.BytesIO(
        doc[page - 1].get_pixmap(dpi=dpi).tobytes("png")))
    W, H = im.size
    _, x1, y1, x2, y2 = box
    return im.crop((int((x1/1000 - pad) * W), int((y1/1000 - pad) * H),
                    int((x2/1000 + pad) * W), int((y2/1000 + pad) * H)))

MIN_FIGURE_AREA = 0.03

MAX_FIGURE_AREA = 0.90

def ocr_elements(ocr_text):
    """The OCR model's regions, in the order it read them: (label, box, text)."""
    marks = list(re.finditer(
        r"(\w+)\s*\[\[\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\s*\]\]", ocr_text))
    found = []
    for i, mark in enumerate(marks):
        stop = marks[i + 1].start() if i + 1 < len(marks) else len(ocr_text)
        box = ("region", *(int(mark.group(k)) for k in range(2, 6)))
        found.append((mark.group(1), box, ocr_text[mark.end():stop].strip()))
    return found

def page_elements(doc, page, transcript=None, dpi=150):
    """A page as ("text", str) and ("figure", Image) pieces, in reading order."""
    if transcript is not None:                    # a scan: already in order
        pieces = []
        for label, box, text in ocr_elements(transcript):
            if label == "image":
                pieces.append(("figure", crop_region(doc, page, box, dpi=dpi)))
            elif text:
                pieces.append(("text", text))
        return pieces

    pg = doc[page - 1]                            # born-digital: sort by
    page_area = pg.rect.width * pg.rect.height    # where things sit
    placed = []
    for x0, y0, x1, y1, text, _, kind in pg.get_text("blocks", sort=True):
        if kind == 0 and text.strip():
            placed.append((y0, ("text", text.strip())))
    for xref, *_ in pg.get_images(full=True):
        rects = pg.get_image_rects(xref)
        if not rects:
            continue
        share = max((r.width * r.height) / page_area for r in rects)
        if not MIN_FIGURE_AREA <= share <= MAX_FIGURE_AREA:
            continue
        # extract_image gives the bytes as stored, whatever the colour depth:
        # line art at 1 bit per pixel comes back as readily as a photograph.
        figure = Image.open(io.BytesIO(doc.extract_image(xref)["image"]))
        placed.append((rects[0].y0, ("figure", figure)))
    return [piece for _, piece in sorted(placed, key=lambda item: item[0])]

def show_elements(pieces):
    for kind, value in pieces:
        if kind == "text":
            print(f"  text   {' '.join(value.split())[:72]}")
        else:
            print(f"  figure {value.size[0]}x{value.size[1]}")

def process_pdf(path, out_dir="processed", needs_ocr=False):
    """Turn a PDF into a folder of markdown pages you can open and read."""
    name = os.path.splitext(os.path.basename(path))[0]
    folder = os.path.join(out_dir, name)
    os.makedirs(os.path.join(folder, "figures"), exist_ok=True)

    doc = open_pdf(path)
    for page in range(1, doc.page_count + 1):
        page_file = os.path.join(folder, f"page-{page:03d}.md")
        if os.path.exists(page_file):
            continue                      # done on an earlier run

        transcript = ocr_page(doc, page) if needs_ocr else None
        lines, figures = [], 0
        for kind, value in page_elements(doc, page, transcript):
            if kind == "text":
                lines.append(value)
            else:
                figures += 1
                relative = f"figures/p{page:03d}-fig{figures:02d}.png"
                value.save(os.path.join(folder, relative))
                lines.append(f"![figure]({relative})")
        with open(page_file, "w") as fh:
            fh.write(f"# page {page}\n\n" + "\n\n".join(lines) + "\n")
        print(f"  page {page}: {figures} figure(s)")
    return folder

def load_pages(folder, pages=None):
    """A processed folder back as (text, figures), ready to send to a model.

    Ollama attaches images to a message rather than placing them in the text,
    so each figure link becomes a numbered marker where it stood, and the
    figures are handed over in that same order.
    """
    wanted = None if pages is None else {int(p) for p in pages}
    chunks, figures = [], []

    for page_file in sorted(glob.glob(os.path.join(folder, "page-*.md"))):
        page = int(os.path.basename(page_file)[5:8])
        if wanted is not None and page not in wanted:
            continue
        with open(page_file) as fh:
            text = fh.read()

        def mark(match):
            figures.append(Image.open(os.path.join(folder, match.group(1))))
            return f"[figure {len(figures)} appears here]"

        chunks.append(re.sub(r"!\[[^\]]*\]\(([^)]+)\)", mark, text))
    return "\n\n".join(chunks), figures

SPECIES_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "author": {"type": "string"},
        "is_new": {"type": "boolean"},
        "max_length": {"type": "number"},
        "n_photos": {"type": "integer"},
        "n_in_photo": {"type": "integer"},
        "synonyms": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["name", "author"],
    "additionalProperties": False
}

def parse_json(text):
    """The JSON in a reply, even if the model wrapped it in something else."""
    text = (text or "").strip()
    if not text:
        raise ValueError("the model returned nothing to parse -- it may have "
                         "spent the whole reply reasoning")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            raise ValueError(f"no JSON in the reply: {text[:120]!r}") from None
        return json.loads(text[start:end + 1])

PAPER_SCHEMA = {
    "type": "object",
    "properties": {
        "species": {"type": "array", "items": SPECIES_SCHEMA}
    },
    "required": ["species"],
    "additionalProperties": False,
}

def as_image_data(fig, max_side=1024):
    """A figure, shrunk if it is huge, encoded the way a model wants it."""
    small = fig.copy()
    small.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    small.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

WHOLE_PAPER_PROMPT = (
    '''List every species this paper describes, using both the text and the figures.
    name is the species name.
    author is the taxonomic authority for that name -- the person who described it -- not the author of this paper; for a species described as new here, use the paper's own authors.
    max_length is the largest body length in mm given for the species.
    is_new is whether the species is described here
    n_photos is the number of photos used to illustrate the species
    synonyms are all synonyms mentioned in the text
    Do not invent values that are not stated.

    '''
)

JSON_ONLY = (
    "\n\nReply with JSON only -- no prose, no markdown, no code fence -- "
    "matching exactly this schema:\n{schema}\n\n"
)

def extract(text, schema=PAPER_SCHEMA, prompt=WHOLE_PAPER_PROMPT, figures=None,
            system=None, model=CHAT_MODEL, client=None, num_ctx=16384,
            schema_in_prompt=False):
    """Text (and figures) in, structured data out.

    client is None for the server on this machine. Session 5 passes a client
    pointed at Ollama's servers instead, which is the only change needed to
    run any of this on a far bigger model.

    system sets the role the model should answer in, the way session 1 did.

    schema_in_prompt writes the schema into the prompt as well as passing it as
    format=. Locally that is redundant -- format= is enforced. Ollama's cloud
    does not enforce it, so there asking is all you have.
    """
    if schema_in_prompt:
        prompt = prompt + JSON_ONLY.format(schema=json.dumps(schema, indent=1))

    message = {"role": "user", "content": prompt + text}
    if figures:
        message["images"] = [as_image_data(fig) for fig in figures]

    messages = [{"role": "system", "content": system}] if system else []
    messages.append(message)

    chat = (client or ollama).chat
    reply = chat(
        model=model,
        messages=messages,
        format=schema,
        think=False,
        options={"temperature": 0, "num_ctx": num_ctx},
    )
    return parse_json(reply.message.content)

def to_table(data, key="species"):
    """The array of records inside an extraction result, as a table.

    Our schemas wrap the records in one named array -- "species" in
    PAPER_SCHEMA. Pass key= if you named yours something else.
    """
    return pd.DataFrame(data.get(key, []))

TOOLS = [
    {"type": "function",
     "function": {
         "name": "get_page_text",
         "description": ("Return the PDF's embedded text layer for a page. "
                         "Free and instant, but on scanned documents it can be "
                         "poor-quality OCR with garbled words."),
         "parameters": {"type": "object",
                        "properties": {"page": {"type": "integer",
                                                "description": "1-based page number"}},
                        "required": ["page"]}}},
    {"type": "function",
     "function": {
         "name": "ocr_page",
         "description": ("Re-read a page from its image with a dedicated OCR "
                         "model. Slower, far more accurate on scans."),
         "parameters": {"type": "object",
                        "properties": {"page": {"type": "integer",
                                                "description": "1-based page number"}},
                        "required": ["page"]}}},
]

def run_tool_loop(messages, tools, impls, think=True, max_turns=6):
    """Let the model call tools until it has an answer.

    Returns (answer, calls_it_made).
    """
    messages = list(messages)
    calls = []
    for _ in range(max_turns):
        reply = ollama.chat(model=CHAT_MODEL, messages=messages, tools=tools,
                            think=think, options={"temperature": 0})
        msg = reply.message
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": msg.tool_calls or []})
        if not msg.tool_calls:
            return (msg.content or ""), calls
        for call in msg.tool_calls:
            name = call.function.name
            args = dict(call.function.arguments)
            calls.append((name, args))
            try:
                result = impls[name](**args)
            except Exception as exc:
                result = f"ERROR: {exc}"        # tell the model, don't crash
            messages.append({"role": "tool", "name": name,
                             "content": str(result)[:6000]})
    return f"stopped after {max_turns} turns", calls

## 1. Your documents

Click the folder icon 📁 in the left sidebar and upload **three to five PDFs**
into `my_pdfs/`. Start small — a vague prompt over four thousand documents is an
expensive way to find out the prompt was vague.

Run the cell below. It prints a sample of the text each PDF carries, so you can
judge it yourself — which is session 2's whole point. A scan usually *has* text;
it is just the output of an OCR pass from years ago, and it can be wrong without
saying so. Read the samples and look for the tell: impossible spellings, letters
swapped for punctuation, words run together.

In [ ]:
MY_FOLDER = "my_pdfs"          # where your PDFs are

os.makedirs(MY_FOLDER, exist_ok=True)
my_papers = sorted(glob.glob(os.path.join(MY_FOLDER, "*.pdf")))

if not my_papers:
    print(f"No PDFs in {MY_FOLDER}/ yet. Upload some with the folder icon "
          f"in the sidebar, then run this cell again.")
for path in my_papers:
    doc = open_pdf(path)
    sample = get_page_text(doc, min(2, doc.page_count))[:400]
    print(f"=== {os.path.basename(path)} ({doc.page_count} pages) ===")
    print(sample.strip() or "(no text at all -- this one is certainly a scan)")
    print()

### Which ones do you not trust?

List the files whose text looked wrong. Those get re-read page by page with the
OCR model; the rest are taken as they are. A file with no text at all always
gets re-read, whatever you say here.

Leave the list empty if they all looked clean — then this is quick.

In [ ]:
RESCAN = [
    # "the_1929_scan.pdf",       # <- file names, one per line
]

my_folders = {}
for path in my_papers:
    name = os.path.basename(path)
    doc = open_pdf(path)
    empty = len(get_page_text(doc, min(2, doc.page_count)).strip()) < 100
    needs_ocr = name in RESCAN or empty
    print(f"{name}: {'re-reading every page' if needs_ocr else 'using the stored text'}"
          f"{' (nothing there to use)' if empty and name not in RESCAN else ''}")
    my_folders[path] = process_pdf(path, needs_ocr=needs_ocr)

if my_folders:
    print("\nwritten to:", *my_folders.values(), sep="\n  ")

## 2. The system prompt

Who should the model be while it reads? This is the role, not the task —
session 1's point that these are role-playing machines. Be specific about the
field and about care: a model told it is a careful taxonomist behaves
differently from one told nothing.

In [ ]:
MY_SYSTEM = (
    "You are a careful DESCRIBE THE SPECIALITY HERE. You read primary "
    "literature and record only what the text actually says. When something "
    "is not stated, you leave it out rather than guessing."
)

## 3. The user prompt

What do you want out of each document? Say what to extract, and say what every
field of your schema means — a field you leave unexplained is a field the model
will fill with something plausible. This is where "sp. n." ended up in an author
column in session 3.

In [ ]:
MY_PROMPT = (
    "DESCRIBE WHAT TO EXTRACT HERE.\n"
    "FIELD_ONE is ...\n"
    "FIELD_TWO is ...\n"
    "FIELD_THREE is ...\n"
    "Do not invent values that are not stated in the text.\n\n"
)

## 4. The schema

The fields you want, and their types. Two levels, as in session 3: one record,
then an array of them, because a document holds many.

Types are `"string"`, `"number"`, `"integer"`, `"boolean"`, and `"array"`. Put
in `required` only the fields that must always be there.

**Tip:** writing schemas by hand is fiddly. Paste yours into Claude or ChatGPT
and ask it to check the JSON schema is valid.

In [ ]:
MY_ITEM_SCHEMA = {                 # one record
    "type": "object",
    "properties": {
        "FIELD_ONE":   {"type": "string"},
        "FIELD_TWO":   {"type": "string"},
        "FIELD_THREE": {"type": "number"},
    },
    "required": ["FIELD_ONE"],
    "additionalProperties": False,
}

MY_SCHEMA = {                      # a document holds many of them
    "type": "object",
    "properties": {
        "records": {"type": "array", "items": MY_ITEM_SCHEMA}
    },
    "required": ["records"],
    "additionalProperties": False,
}

## 5. Run it

Each processed document goes to the model with your system prompt, your user
prompt and your schema, and the rows come back into one table with a `source`
column saying which paper each came from.

Then **read the table against the papers**. The shape is guaranteed; the
content is not. That has been the whole point of the day.

In [ ]:
if not my_papers:
    print(f"Upload PDFs into {MY_FOLDER}/ and run step 1 first.")
else:
    frames = []
    for path, folder in my_folders.items():
        name = os.path.basename(path)
        text, figures = load_pages(folder)
        print(f"{name}: {len(text)} characters, {len(figures)} figures")
        try:
            data = extract(text[:30000], schema=MY_SCHEMA, prompt=MY_PROMPT,
                           figures=figures[:8], system=MY_SYSTEM)
        except Exception as exc:
            print(f"   FAILED -- {exc}")
            continue
        rows = to_table(data, key="records")
        print(f"   {len(rows)} record(s)")
        if len(rows):
            rows.insert(0, "source", name)
            frames.append(rows)

    my_table = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    my_table.to_csv("my_results.csv", index=False)
    print(f"\n{len(my_table)} rows, saved to my_results.csv")
    display(my_table)